In [1]:
import sys
import subprocess
import pandas as pd
import numpy as np
import requests
from dotenv import load_dotenv
import os
from sqlalchemy import create_engine
import tweepy
import openai
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import xgboost as xgb
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import streamlit as st

# Step 0: Connecting to SQL Database

In [ ]:
# Verify Database and proper tokens exist in .env file
from dotenv import load_dotenv

load_dotenv()

print("Database URL found:", os.getenv("DATABASE_URL") is not None)
print("Twitter token found:", os.getenv("X_BEARER_TOKEN") is not None)

Database URL found: True
Twitter token found: True


In [ ]:
# Connect to the database and run a simple query to verify connection
from sqlalchemy import create_engine, text

engine = create_engine(os.getenv("DATABASE_URL"))

with engine.connect() as conn:
    result = conn.execute(
        text("SELECT current_database(), current_user")
    )
    
    print(result.fetchone())

('twitter_engagement', 'postgres')


In [ ]:
# Verify tables exist in the database
query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name
"""

with engine.connect() as conn:
    result = conn.execute(text(query))

    for row in result:
        print(row[0])

tweet_labels
tweet_media
tweets
users


# Step 1: Getting Twitter Data

In [5]:
import os
import re
import json
import pandas as pd
import tweepy

from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()

engine = create_engine(os.getenv("DATABASE_URL"))

client = tweepy.Client(
    bearer_token=os.getenv("X_BEARER_TOKEN"),
    wait_on_rate_limit=True
)

print("Setup complete")

Setup complete


In [6]:
# Text cleaning function
def clean_tweet_text(text):
    if text is None:
        return None
    
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "@USER", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"\s+", " ", text)
    
    return text.strip()

In [15]:
# Test collection
from datetime import datetime, timedelta, timezone

now = datetime.now(timezone.utc)

start_time = now - timedelta(days=7)
end_time = now - timedelta(days=6)

response = client.search_recent_tweets(
    query='("the" OR "this" OR "you" OR "that" OR "with") lang:en -is:retweet',
    max_results=10,
    start_time=start_time.isoformat(timespec="seconds").replace("+00:00", "Z"),
    end_time=end_time.isoformat(timespec="seconds").replace("+00:00", "Z"),
    tweet_fields=[
        "created_at",
        "author_id",
        "lang",
        "public_metrics",
        "entities",
        "attachments",
        "referenced_tweets"
    ],
    expansions=[
        "attachments.media_keys",
        "author_id",
        "referenced_tweets.id"
    ],
    media_fields=[
        "media_key",
        "type",
        "url",
        "preview_image_url",
        "alt_text",
        "width",
        "height",
        "duration_ms",
        "public_metrics"
    ],
    user_fields=[
        "username",
        "name",
        "verified",
        "public_metrics"
    ]
)

print("Tweets returned:", 0 if response.data is None else len(response.data))
print(response.data[0].data if response.data else "No tweets returned")

Tweets returned: 6
{'entities': {'mentions': [{'start': 154, 'end': 170, 'username': 'linglingsirilak', 'id': '1202752584416514048'}], 'urls': [{'start': 171, 'end': 194, 'url': 'https://t.co/jPUJphbrvC', 'expanded_url': 'https://x.com/mia_freedom/status/2062376322303984038/photo/1', 'display_url': 'pic.x.com/jPUJphbrvC', 'media_key': '3_2062376301009465344'}], 'hashtags': [{'start': 97, 'end': 124, 'tag': 'CLINIQUExLinglingInBeijing'}, {'start': 125, 'end': 139, 'tag': 'Linglingkwong'}, {'start': 140, 'end': 153, 'tag': 'หลิงหลิงคอง\u200c'}]}, 'created_at': '2026-06-04T03:30:06.000Z', 'id': '2062376322303984038', 'author_id': '49399753', 'text': 'The one and only appears regal representing Thailand 📍 Global star! \n\nLING PRESENTER OF CLINIQUE\n#CLINIQUExLinglingInBeijing\n#Linglingkwong #หลิงหลิงคอง\u200c\n@linglingsirilak https://t.co/jPUJphbrvC', 'lang': 'en', 'public_metrics': {'retweet_count': 0, 'reply_count': 0, 'like_count': 0, 'quote_count': 0, 'bookmark_count': 0, 'impressio

In [19]:
# Dataframe creation
tweets = response.data or []
includes = response.includes or {}

media_lookup = {
    media["media_key"]: media
    for media in includes.get("media", [])
}

tweet_rows = []
media_rows = []

for tweet in tweets:
    metrics = tweet.public_metrics or {}
    attachments = tweet.data.get("attachments", {})
    media_keys = attachments.get("media_keys", [])

    tweet_rows.append({
        "tweet_id": str(tweet.id),
        "author_id": str(tweet.author_id),
        "created_at": tweet.created_at,
        "text": tweet.text,
        "clean_text": clean_tweet_text(tweet.text),
        "lang": tweet.lang,
        "retweet_count": metrics.get("retweet_count", 0),
        "reply_count": metrics.get("reply_count", 0),
        "like_count": metrics.get("like_count", 0),
        "quote_count": metrics.get("quote_count", 0),
        "has_media": len(media_keys) > 0,
        "media_count": len(media_keys),
        "collection_source": "broad_search",
        "search_query": query,
        "raw_json": json.dumps(tweet.data)
    })

    for media_key in media_keys:
        media = media_lookup.get(media_key)

        if media is None:
            continue

        media_rows.append({
            "media_key": media_key,
            "tweet_id": str(tweet.id),
            "media_type": media.get("type"),
            "url": media.get("url"),
            "preview_image_url": media.get("preview_image_url"),
            "alt_text": media.get("alt_text"),
            "width": media.get("width"),
            "height": media.get("height"),
            "duration_ms": media.get("duration_ms"),
            "public_metrics": json.dumps(media.get("public_metrics", {})),
            "raw_json": json.dumps(media.data)
        })

tweets_df = pd.DataFrame(tweet_rows)
media_df = pd.DataFrame(media_rows)

tweets_df["engagement_total"] = (
    tweets_df["like_count"] +
    tweets_df["reply_count"] +
    tweets_df["retweet_count"] +
    tweets_df["quote_count"]
)

# sort tweets_df by engagement_total in descending order
tweets_df = tweets_df.sort_values(by="engagement_total", ascending=False)

display(tweets_df.head())
display(media_df.head())

,tweet_id,author_id,created_at,text,clean_text,lang,retweet_count,reply_count,like_count,quote_count,has_media,media_count,collection_source,search_query,raw_json,engagement_total
4,2062376322060714298,17629557,2026-06-04 03:30:06+00:00,@ThoNg676733 @IntheFrame1 tell me this does no...,@USER @USER tell me this does not have major M...,en,0,1,1,0,False,0,broad_search,"(""the"" OR ""this"" OR ""you"" OR ""that"" OR ""with"")...","{""entities"": {""mentions"": [{""start"": 0, ""end"":...",2
1,2062376322186555890,72459373,2026-06-04 03:30:06+00:00,"When your pizza is hot, the SARAP hits differe...","When your pizza is hot, the SARAP hits differe...",en,0,0,1,0,True,4,broad_search,"(""the"" OR ""this"" OR ""you"" OR ""that"" OR ""with"")...","{""created_at"": ""2026-06-04T03:30:06.000Z"", ""id...",1
5,2062376322043683136,14464882,2026-06-04 03:30:06+00:00,Touching up Tim Currys makeup on the set of It...,Touching up Tim Currys makeup on the set of It,en,0,0,1,0,True,1,broad_search,"(""the"" OR ""this"" OR ""you"" OR ""that"" OR ""with"")...","{""created_at"": ""2026-06-04T03:30:06.000Z"", ""id...",1
2,2062376322157121845,755694274004578304,2026-06-04 03:30:06+00:00,@JapaneseCr53923 @114514yk @Based_lalafell @0x...,@USER @USER @USER @USER @USER If thats your pr...,en,0,1,0,0,False,0,broad_search,"(""the"" OR ""this"" OR ""you"" OR ""that"" OR ""with"")...","{""entities"": {""mentions"": [{""start"": 0, ""end"":...",1
0,2062376322303984038,49399753,2026-06-04 03:30:06+00:00,The one and only appears regal representing Th...,The one and only appears regal representing Th...,en,0,0,0,0,True,1,broad_search,"(""the"" OR ""this"" OR ""you"" OR ""that"" OR ""with"")...","{""entities"": {""mentions"": [{""start"": 154, ""end...",0


,media_key,tweet_id,media_type,url,preview_image_url,alt_text,width,height,duration_ms,public_metrics,raw_json
0,3_2062376301009465344,2062376322303984038,photo,https://pbs.twimg.com/media/HJ8IUZ7acAAm_T9.jpg,None,None,1024,1820,None,null,"{""height"": 1820, ""url"": ""https://pbs.twimg.com..."
1,3_2062372301031841792,2062376322186555890,photo,https://pbs.twimg.com/media/HJ8Erk2boAA7Gle.jpg,None,None,1080,1080,None,null,"{""height"": 1080, ""url"": ""https://pbs.twimg.com..."
2,3_2062372329343299584,2062376322186555890,photo,https://pbs.twimg.com/media/HJ8EtOUaMAA-f3n.jpg,None,None,1080,1080,None,null,"{""height"": 1080, ""url"": ""https://pbs.twimg.com..."
3,3_2062372699977216000,2062376322186555890,photo,https://pbs.twimg.com/media/HJ8FCzCa8AAIQlw.jpg,None,None,1080,1080,None,null,"{""height"": 1080, ""url"": ""https://pbs.twimg.com..."
4,3_2062375560148586496,2062376322186555890,photo,https://pbs.twimg.com/media/HJ8HpSAakAAKLSm.jpg,None,None,1080,1080,None,null,"{""height"": 1080, ""url"": ""https://pbs.twimg.com..."
